In [23]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark import SparkFiles
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.regression import LinearRegression

import pandas as pd

os.environ["HADOOP_HOME"] = r"C:\hadoop\winutils\hadoop-3.3.6"
os.environ["hadoop.home.dir"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" + os.environ["PATH"]

# Inicializar sesión
spark = SparkSession.builder.appName("RegresionStartups").getOrCreate()

In [24]:
# rescatamos el dataframe del fichero csv del titanic
titanic_df = spark.read.csv("./open-datasets/titanic.csv", header=True, inferSchema=True)
titanic_df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [25]:
# 1. Convertir la columna de texto 'Sex' (male/female) a índice numérico (0.0 / 1.0)
indexer = StringIndexer(inputCol="Sex", outputCol="Sex_index")
df_indexado = indexer.fit(titanic_df).transform(titanic_df)

# 2. Agrupar las variables independientes en una única columna llamada 'features'
assembler = VectorAssembler(
inputCols=["Pclass", "Sex_index", "Age", "Fare"],
outputCol="features"
)
df_final = assembler.transform(df_indexado)

# 3. Dividir el dataset: 80% para entrenar el modelo y 20% para evaluarlo
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

In [26]:
# 1. Inicializar el clasificador con 50 árboles de decisión
rf = RandomForestClassifier(featuresCol="features", labelCol="Survived", numTrees=50, maxDepth=5, seed=42)

# 2. Entrenar el modelo con los datos de entrenamiento
# Ignorar las filas que contienen valores nulos en las columnas de entrada
assembler.setHandleInvalid("skip")

# Volver a crear el vector de características y dividir los datos
df_final = assembler.transform(df_indexado)
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Entrenar el modelo
modelo_rf = rf.fit(train_data)

# 3. Realizar predicciones sobre los datos de prueba
predicciones = modelo_rf.transform(test_data)

# Mostrar resultados: 'probability' muestra la certeza del modelo (ej. 70% vivo, 30% fallecido)
predicciones.select("Pclass", "Sex_index", "Age", "Survived", "prediction", "probability").show(5)

# 4. Calcular la precisión, recall y F1 score global del modelo
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="f1")

accuracy = evaluator_accuracy.evaluate(predicciones)
precision = evaluator_precision.evaluate(predicciones)
recall = evaluator_recall.evaluate(predicciones)
f1 = evaluator_f1.evaluate(predicciones)

print(f"Precisión (Accuracy) del Random Forest: {accuracy * 100:.2f}%")
print(f"Precisión (Precision) del Random Forest: {precision * 100:.2f}%")
print(f"Recall del Random Forest: {recall * 100:.2f}%")
print(f"F1 Score del Random Forest: {f1 * 100:.2f}%")

+------+---------+----+--------+----------+--------------------+
|Pclass|Sex_index| Age|Survived|prediction|         probability|
+------+---------+----+--------+----------+--------------------+
|     3|      1.0|26.0|       1|       1.0|[0.46861227790166...|
|     3|      0.0| 2.0|       0|       0.0|[0.55469005958301...|
|     2|      1.0|14.0|       1|       1.0|[0.10830020657649...|
|     3|      1.0|14.0|       0|       1.0|[0.32177849199247...|
|     3|      1.0|15.0|       1|       1.0|[0.36601975371455...|
+------+---------+----+--------+----------+--------------------+
only showing top 5 rows
Precisión (Accuracy) del Random Forest: 78.95%
Precisión (Precision) del Random Forest: 78.70%
Recall del Random Forest: 78.95%
F1 Score del Random Forest: 78.72%


## Explicación de las métricas de clasificación

El modelo intenta predecir `Survived`:

- `0`: la persona no sobrevivió.
- `1`: la persona sobrevivió.

Para entender las métricas, primero hay que comparar dos valores de cada pasajero:

- `Survived`: valor real del dataset.
- `prediction`: clase predicha por el modelo.

### Matriz de confusión

La matriz de confusión cuenta aciertos y errores:

| Valor real | Predicción | Nombre | Interpretación en este ejercicio |
|---:|---:|---|---|
| 0 | 0 | Verdadero negativo (`TN`) | Predijo que no sobrevivía y acertó |
| 0 | 1 | Falso positivo (`FP`) | Predijo supervivencia, pero no sobrevivió |
| 1 | 0 | Falso negativo (`FN`) | Sobrevivió, pero el modelo no lo detectó |
| 1 | 1 | Verdadero positivo (`TP`) | Predijo supervivencia y acertó |

En este contexto, positivo significa pertenecer a la clase `1`, es decir, sobrevivir. La palabra “positivo” no significa necesariamente algo bueno; solo identifica la clase que estamos tomando como referencia.

### Accuracy

`Accuracy` mide la proporción total de predicciones correctas:

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

En código:

```python
accuracy = evaluator_accuracy.evaluate(predicciones)
```

Ejemplo: si el modelo acierta 80 pasajeros de 100, su `accuracy` es `0.80`, o `80 %`.

Es fácil de interpretar, pero puede ser engañosa cuando las clases están desbalanceadas. Si el 80 % de los pasajeros no sobreviviera, un modelo que siempre predijera `0` tendría `80 %` de accuracy, aunque no detectaría ningún superviviente.

### Precision

`Precision` responde a esta pregunta:

> De todas las personas que el modelo predijo como supervivientes, ¿cuántas sobrevivieron realmente?

$$
Precision = \frac{TP}{TP + FP}
$$

Un ejemplo:

```text
El modelo predice 20 supervivientes.
De esos 20, realmente sobreviven 15.
Precision = 15 / 20 = 0.75
```

Una precision alta significa que cuando el modelo dice `prediction=1.0`, normalmente acierta. Penaliza los falsos positivos.

### Recall

`Recall`, también llamado sensibilidad, responde a esta pregunta:

> De todas las personas que realmente sobrevivieron, ¿cuántas detectó el modelo?

$$
Recall = \frac{TP}{TP + FN}
$$

Ejemplo:

```text
Realmente sobreviven 30 personas.
El modelo detecta 18.
Recall = 18 / 30 = 0.60
```

Un recall alto significa que se escapan pocos casos positivos. Penaliza los falsos negativos.

### F1 Score

`F1` combina `precision` y `recall` mediante la media armónica:

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

La media armónica hace que el resultado sea bajo si una de las dos métricas es muy baja. Por eso F1 es útil cuando queremos equilibrio:

- Precision alta y recall bajo: el modelo es muy selectivo, pero se le escapan muchos supervivientes.
- Recall alto y precision baja: detecta muchos supervivientes, pero genera muchos falsos positivos.
- F1 alto: busca un equilibrio entre ambos objetivos.

### Qué significan las métricas del código

El notebook calcula:

```python
evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="weightedRecall"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="f1"
)
```

`accuracy` calcula el porcentaje global de aciertos. En cambio, `weightedPrecision`, `weightedRecall` y `f1` tienen en cuenta el rendimiento de las clases ponderándolo por el número de ejemplos de cada una.

Una métrica ponderada puede parecer buena porque la clase mayoritaria tiene mucho peso. Para conocer específicamente la capacidad de detectar supervivientes, conviene calcular también las métricas de la clase `1` sin quedarse únicamente con los valores globales.

### Analizar la matriz de confusión con Spark

```python
predicciones.groupBy(
    "Survived", "prediction"
).count().orderBy(
    "Survived", "prediction"
).show()
```

La salida permite localizar directamente los cuatro casos:

```text
Survived=0, prediction=0 -> TN
Survived=0, prediction=1 -> FP
Survived=1, prediction=0 -> FN
Survived=1, prediction=1 -> TP
```

### Cómo elegir la métrica

La métrica más importante depende del objetivo:

- Si importa acertar globalmente, observar `accuracy`.
- Si las predicciones positivas deben ser muy fiables, observar `precision`.
- Si es importante no dejar escapar casos positivos, observar `recall`.
- Si se necesita equilibrio entre precision y recall, observar `F1`.

En un proyecto real no se debe elegir un modelo solo porque tenga el porcentaje más alto de accuracy. Hay que estudiar la matriz de confusión, la distribución de las clases y el coste de cada tipo de error.

In [27]:
# Matriz de confusión: una sola agregación en Spark y visualización clara con pandas
from IPython.display import display

# Contar cada combinación de valor real y predicción.
# collect() es suficiente porque solo devuelve como máximo cuatro combinaciones.
conteos_spark = (
    predicciones
    .groupBy("Survived", "prediction")
    .count()
    .collect()
)

# Construir una tabla 2x2 aunque alguna combinación no aparezca en los datos.
clases = [0, 1]
matriz_conteos = pd.DataFrame(
    0,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

for fila in conteos_spark:
    clase_real = int(fila["Survived"])
    clase_predicha = int(fila["prediction"])
    matriz_conteos.loc[
        f"Actual {clase_real}",
        f"Predicted {clase_predicha}"
    ] = int(fila["count"])

print("Matriz de confusión: conteos")
display(
    matriz_conteos.style
    .background_gradient(cmap="Blues")
    .format("{:,.0f}")
)

# Porcentajes dentro de cada clase real: ayuda a interpretar recall por clase.
matriz_porcentajes = matriz_conteos.div(
    matriz_conteos.sum(axis=1).replace(0, 1),
    axis=0
)

print("Matriz de confusión: porcentajes por clase real")
display(
    matriz_porcentajes.style
    .background_gradient(cmap="Greens", vmin=0, vmax=1)
    .format("{:.1%}")
)

# Nombrar explícitamente los cuatro tipos de resultado.
tn = matriz_conteos.loc["Actual 0", "Predicted 0"]
fp = matriz_conteos.loc["Actual 0", "Predicted 1"]
fn = matriz_conteos.loc["Actual 1", "Predicted 0"]
tp = matriz_conteos.loc["Actual 1", "Predicted 1"]

casos = pd.DataFrame({
    "Caso": ["TN", "FP", "FN", "TP"],
    "Descripcion": [
        "No sobrevivió y el modelo predijo 0",
        "No sobrevivió y el modelo predijo 1",
        "Sobrevivió y el modelo predijo 0",
        "Sobrevivió y el modelo predijo 1"
    ],
    "Cantidad": [tn, fp, fn, tp]
})

print("Detalle de aciertos y errores")
display(casos.style.format({"Cantidad": "{:,.0f}"}))

Matriz de confusión: conteos


,Predicted 0,Predicted 1
Actual 0,61,10
Actual 1,14,29


Matriz de confusión: porcentajes por clase real


,Predicted 0,Predicted 1
Actual 0,85.9%,14.1%
Actual 1,32.6%,67.4%


Detalle de aciertos y errores


,Caso,Descripcion,Cantidad
0,TN,No sobrevivió y el modelo predijo 0,61
1,FP,No sobrevivió y el modelo predijo 1,10
2,FN,Sobrevivió y el modelo predijo 0,14
3,TP,Sobrevivió y el modelo predijo 1,29
